# Dice Bingo

**Authors**: David J. Hemmer and Benjamin W. Ong, Michigan Technological University

**License:** [MIT License](https://opensource.org/licenses/MIT)

This work is licensed under the MIT License. See the LICENSE file for details.

## Game Overview

Dice Bingo is a game played on a $3\times 3$ board filled with integers from 2 to 12, representing possible outcomes of rolling two standard dice.

At each turn, two dice are rolled and their sum is computed. If that sum appears on any unmarked cell of the board, the player may mark one such cell. If multiple matching cells exist, the player chooses one according to a strategy.

The goal is to complete a full winning line—any row, column, or diagonal—by marking all three cells in that line. The game ends immediately when a winning line is completed.

This notebook analyzes the game using exact probability calculations and optimal decision-making:
- Computes optimal play strategy for a single board
- Computes expected time to win for a single board
- Analyzes head-to-head competition between two boards using shared dice rolls, independent strategy
- Verifies a nontransitive set of boards, where board $A$ beats board $B$, board $B$ beats board $C$, and board $C$ beats board $A$

## Imports and global constants

We will use the fractions package to compute probabilities in exact arithmetic. We also specify some useful global constants:
- `DICE_PROB`: probability of rolling dice sums
- `WINNING_LINES`: specifying which combination of squares form a bingo

In [ ]:
from fractions import Fraction
import matplotlib.pyplot as plt

# Winning line indices for a 3x3 board
WINNING_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8), # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8), # columns
    (0, 4, 8), (2, 4, 6),            # diagonals
]

# Dice probabilities for two dice
DICE_PROB = {
    2: Fraction(1, 36), 3: Fraction(2, 36), 4: Fraction(3, 36),
    5: Fraction(4, 36), 6: Fraction(5, 36), 7: Fraction(6, 36),
    8: Fraction(5, 36), 9: Fraction(4, 36), 10: Fraction(3, 36),
    11: Fraction(2, 36), 12: Fraction(1, 36),
}

## Representing Boards Using Bits

Each 3x3 Dice Bingo board is represented as a **9-bit integer**, where each bit corresponds to a cell on the board:  

- **Bit k = 0** → cell k is unmarked  
- **Bit k = 1** → cell k is marked  

This approach is very efficient because:  

1. **Compact representation:** One integer stores the entire board state.  
2. **Fast operations:**  
   - Marking a cell → `mask | (1 << i)`  
   - Checking a winning line → `(mask & winning_mask) == winning_mask`  
   - Counting marked cells → `mask.bit_count()`  
3. **Easy enumeration:** All possible board states (0–511) can be iterated systematically.  

Using bitmasks makes computing expected values and head-to-head probabilities **much faster and simpler**, especially when combined with precomputed winning line masks.  

In [ ]:
# Precompute winning masks
def bit(i): return 1 << i
def line_to_mask(line): return sum(bit(i) for i in line)
WINNING_MASKS = [line_to_mask(line) for line in WINNING_LINES]

def is_winning(mask):
    return any((mask & L) == L for L in WINNING_MASKS)

def mask_size(mask):
    return mask.bit_count()

def mark_cell(mask, i):
    return mask | bit(i)

## Helper functions
We specify some helper functions 
- `check_board`: is this a valid 3x3 board whose cells contains entries $2\le b_i \le 12$
- `candidates`: finds all currently available cells that match a given dice roll value.
- `hitting_sums`: finds all dice sums that could "hit" an unmarked cell on the board for the current state.
- `print_board`: pretty print board
- `transient_mask`: returns all non-winning board states.

In [ ]:
def check_board(board):
    if len(board) != 9:
        raise ValueError("Board must have length 9.")
    for x in board:
        if x < 2 or x > 12:
            raise ValueError("Board entries must be in {2..12}.")
    return tuple(board)

def candidates(board, mask, v):
    return [i for i in range(9) if not (mask & bit(i)) and board[i] == v]

def hitting_sums(board, mask):
    return sorted({board[i] for i in range(9) if not (mask & bit(i))})

def print_board(board):
    board = check_board(board)
    for r in range(3):
        print(board[3*r], board[3*r+1], board[3*r+2])

def transient_masks():
    return [m for m in range(1 << 9) if not is_winning(m)]

## Optimal Solo Strategy
Since the outcome of each roll depends only on the current set of marked squares, and since the player may sometimes choose among several possible marks, the optimal expected time to completion is described by Bellman equations for the corresponding Markov decision process. Let $V(S)$ denote the expected number of rolls from state $S$ to a winning state under optimal play,
\begin{align}
  V(S) = \mathbb{E}[\text{rolls remaining to win | current state } S]
\end{align}
Given state $S$, define the set of \emph{hitting sums}:
\begin{align}
H(S) = \{v \in \{2,\ldots,12\} : \text{there exists } i \notin S \text{ with } b_i = v\}.
\end{align}
These are the dice sums that match at least one unmarked cell.  Any sum $v \notin H(S)$ is _wasted_: the state remains at $S$ if a $v$ is rolled. The probability of a useful roll is
\begin{align}
p_{\mathrm{hit}}(S) = \sum_{v \in H(S)} p(v).
\end{align}
For each hitting sum $v \in H(S)$, the player chooses which cell to mark among the candidates
\begin{align}
C(S, v) = \{i \notin S : b_i = v\}.
\end{align}
Marking cell $c \in C(S,v)$ transitions to state $S \cup \{c\}$.  When there is a choice, the player will mark the cell that minimizes the expected number of future rolls:
\begin{align}
c^*(S,v) = \operatorname*{arg\,min}_{c \,\in\, C(S,v)} V(S \cup \{c\}),
\end{align}
where $V(S \cup \{c\}) = 0$ if $S \cup \{c\} \in \mathcal{W}$.
Finally, the Bellman equation reads:
\begin{align}
V(S) = \frac{1 + \displaystyle\sum_{v \in H(S)} p(v)\, V\!\bigl(S \cup \{c^*(S,v)\}\bigr)}{p_{\mathrm{hit}}(S)}.
\end{align}

To compute this value function $V(S)$, we first observe that marking a cell increases $|S|$ by exactly one: the value $V(S \cup \{c^*\})$ involves a state with $|S|+1$ marked cells.  Therefore, if we process states in decreasing order of $|S|$, every value on the right-hand side is already computed when we evaluate $V(S)$. So we can compute any $V(S)$ via the following recursive procedure, noting that if $|S| \geq 7$ it must already be winning:

1. **Base case**. For all $S \in \mathcal{W}$, set $V(S) = 0$.
2. **Recursive step**. For $k = 6, 5, \ldots, 1, 0$, process each transient state $S$ with $|S| = k$:
    1. Compute $H(S)$ and $p_{\mathrm{hit}}(S)$.
\item For each $v \in H(S)$, find $c^*(S,v)$ by evaluating $V(S \cup \{c\})$ for all $c \in C(S,v)$ and choosing the minimum.  (All these values are already known because $|S \cup \{c\}| = k + 1 > k$.)
    2. Compute $V(S)$ via using the Bellman equation.

3. **Output**. The quantity of interest is $V(\varnothing)$: the expected number of rolls from the initially unmarked board.

We provide two functions:
- `single_board_values_and_strategy`: computes and stores the full strategy and values for all states
- `expected_winning_time`: expected number of dice rolls for an unmarked board to win

In [ ]:
def single_board_values_and_strategy(board, tie_break="smallest"):

    # make sure boards are valid
    board = check_board(board)
    
    # tie_break: what to do if multiple cells give the same expected 
    # time to win, pick either   
    if tie_break not in ("smallest", "largest"):
        raise ValueError("tie_break must be 'smallest' or 'largest'.")
    V, strategy = {}, {}

    # Base case: winning masks
    for mask in range(1 << 9):
        if is_winning(mask): V[mask] = Fraction(0)

    # Backward induction
    for k in reversed(range(7)):
        for mask in range(1 << 9):
            if mask_size(mask) != k or is_winning(mask): continue
            H = hitting_sums(board, mask)
            p_hit = sum(DICE_PROB[v] for v in H)
            future_sum = Fraction(0)

            for v in H:
                C = candidates(board, mask, v)
                future_values = [(V[mark_cell(mask, c)], c) for c in C]
                best_value = min(value for value, c in future_values)
                best_cells = [c for value, c in future_values if value == best_value]
                chosen = min(best_cells) if tie_break=="smallest" else max(best_cells)
                strategy[(mask, v)] = chosen
                future_sum += DICE_PROB[v] * V[mark_cell(mask, chosen)]

            V[mask] = (Fraction(1) + future_sum) / p_hit
    return V, strategy

def expected_winning_time(board, tie_break="smallest"):
    V, _ = single_board_values_and_strategy(board, tie_break=tie_break)
    return V[0]

## Examples: Solo Board Strategy

### Figure 2

Let's compute the expected winning time for the board displayed in Figure 2 of the manuscript
\begin{array}{|c|c|c|}
\hline
6 & 7 & 6 \\ \hline
7 & 7 & 7 \\ \hline
6 & 6 & 6 \\ \hline
\end{array}

In [ ]:
board = [
    6, 7, 6,
    7, 7, 7,
    6, 6, 6,
]

V = expected_winning_time(board,"smallest")
print("V = ",V)
print("V \u2248",float(V))

### Board of all 7s
Section 2.4 in manuscript.
\begin{array}{|c|c|c|}
\hline
7 & 7 & 7 \\ \hline
7 & 7 & 7 \\ \hline
7 & 7 & 7 \\ \hline
\end{array}

In [ ]:
board7 = [
    7, 7, 7,
    7, 7, 7,
    7, 7, 7,
]

V7 = expected_winning_time(board7,"smallest")
print("V = ",V7)
print("V \u2248",float(V7))

### Optimal Board
Section 2.5 in manuscript (up to symmetry)
\begin{array}{|c|c|c|}
\hline
8 & 8 & 9 \\ \hline
7 & 6 & 10 \\ \hline
7 & 4 & 5 \\ \hline
\end{array}

In [ ]:
board_optimal = [
    8, 8, 9,
    7, 6, 10,
    7, 4, 5,
]

V_optimal = expected_winning_time(board_optimal,"smallest")
print("V = ",V_optimal)
print("V \u2248",float(V_optimal))

### Magic Square
All rows, columns and diagonals sum to 21.
\begin{array}{|c|c|c|}
\hline
6 & 5 & 10 \\ \hline
11 & 7 & 3 \\ \hline
4 & 9 & 8 \\ \hline
\end{array}

In [ ]:
board_magic = [
    6, 5, 10,
    11, 7, 3,
    4, 9, 8,
]

V_magic = expected_winning_time(board_magic,"smallest")
print("V = ",V_magic)
print("V \u2248",float(V_magic))

## Head-to-Head 
Suppose two players, each with their own board, observe the same sequence of dice rolls. Each player independently follows their own optimal solo strategy -- players do not see each other's boards and do not adjust their strategies based on their opponent's state. The game ends when at least one player completes a bingo.

Define the random variables
\begin{align*}
  T_A = \text{first roll that player A wins} \\
  T_B = \text{first roll that player B wins.}   
\end{align*}    
Because both players see the same sequence of dice rolls, $T_A$ and $T_B$ are _not_ independent -- their joint distribution is determined by the shared dice sequence.  The three outcomes of interest are
\begin{align*}
P(T_A < T_B), \qquad P(T_B < T_A), \qquad P(T_A = T_B).
\end{align*}
We say **$A$ beats $B$** if $P(T_A < T_B) > P(T_B < T_A)$.  These probabilities cannot be determined only from $V_A$ and $V_B$, the expected number of dice rolls for board $A$ and board $B$ to find a bingo respectively. 
We need the full joint distribution, which we now compute in exact arithmetic using a Markov chain where the states keep track of both boards simultaneously.

There are three functions in this section:
- `next_state_under_strategy`: pick the optimal cell to mark
- `head_to_head_probabilities`: in a head-to-head competition between board $A$ and board $B$, compute probabilities that $T_A>T_B$, $T_B>T_A$, $T_A = T_B$.
- `head_to_head_distribution`: in a head-to-head competition between board$A$ and board $B$, compute probabilities that $T_A = k$, $T_B = k$ at roll $k$

In [ ]:
def next_state_under_strategy(board, strategy, mask, v):
    C = candidates(board, mask, v)
    if not C: return mask
    chosen = strategy[(mask, v)]
    return mark_cell(mask, chosen)

def head_to_head_probabilities(boardA, boardB, tie_break="smallest"):

    # make sure boards are valid
    boardA, boardB = check_board(boardA), check_board(boardB)

    # get optimal strategy for each board for all sattes
    VA, stratA = single_board_values_and_strategy(boardA, tie_break)
    VB, stratB = single_board_values_and_strategy(boardB, tie_break)

    # initialize counters to see who wins
    Awin, Bwin, Tie = {}, {}, {}

    # which states (masks) are transient
    trans = transient_masks()

    # list of all possible joint states
    joint_states = [(mA, mB) for mA in trans for mB in trans]

    #[ongbw]: thinks this should be range (13), since |S| > 12 are absorbing
    for total_size in reversed(range(17)):
        for mA, mB in joint_states:
            # only look at states that have the appropriate total size (backwards induction)
            if mask_size(mA)+mask_size(mB) != total_size: continue

            # initialize
            p_self = rhs_A = rhs_B = rhs_T = Fraction(0)
            for v, pv in DICE_PROB.items():
                newA = next_state_under_strategy(boardA, stratA, mA, v)
                newB = next_state_under_strategy(boardB, stratB, mB, v)
                A_wins_now, B_wins_now = is_winning(newA), is_winning(newB)

                if A_wins_now and B_wins_now: rhs_T += pv
                elif A_wins_now: rhs_A += pv
                elif B_wins_now: rhs_B += pv
                else:
                    if newA==mA and newB==mB: p_self += pv
                    else:
                        rhs_A += pv*Awin.get((newA,newB),0)
                        rhs_B += pv*Bwin.get((newA,newB),0)
                        rhs_T += pv*Tie.get((newA,newB),0)

            denom = Fraction(1) - p_self
            Awin[(mA,mB)] = rhs_A / denom
            Bwin[(mA,mB)] = rhs_B / denom
            Tie[(mA,mB)] = rhs_T / denom

    # return who wins, from empty boards
    return Awin[(0,0)], Bwin[(0,0)], Tie[(0,0)]

def head_to_head_distribution(boardA, boardB, kmax=50, tie_break="smallest"):
    boardA, boardB = check_board(boardA), check_board(boardB)
    VA, stratA = single_board_values_and_strategy(boardA, tie_break)
    VB, stratB = single_board_values_and_strategy(boardB, tie_break)

    current = {(0,0): Fraction(1)}
    distA, distB, distTie = {}, {}, {}

    for k in range(1,kmax+1):
        next_current = {}
        prob_A = prob_B = prob_T = Fraction(0)

        for (mA,mB), state_prob in current.items():
            for v, pv in DICE_PROB.items():
                newA = next_state_under_strategy(boardA,stratA,mA,v)
                newB = next_state_under_strategy(boardB,stratB,mB,v)
                transition_prob = state_prob*pv
                A_wins_now, B_wins_now = is_winning(newA), is_winning(newB)

                if A_wins_now and B_wins_now: prob_T += transition_prob
                elif A_wins_now: prob_A += transition_prob
                elif B_wins_now: prob_B += transition_prob
                else:
                    next_current[(newA,newB)] = next_current.get((newA,newB),Fraction(0))+transition_prob

        distA[k], distB[k], distTie[k] = prob_A, prob_B, prob_T
        current = next_current

    return distA, distB, distTie

## Examples: head-to-head

### An unexpected result
It is possible that a board with a smaller expected winning time is **not** favored in a head to head matchup.  
(Section 4.4 in manuscript)
\begin{align}
\text{Board }A:
\begin{array}{|c|c|c|}
\hline
9 & 6 & 7 \\ \hline
7 & 9 & 6 \\ \hline
6 & 7 & 9 \\ \hline
\end{array}
\qquad
\text{Board }B: 
\begin{array}{|c|c|c|}
\hline
9 & 6 & 7 \\ \hline
7 & 9 & 6 \\ \hline
6 & 7 & 9 \\ \hline
\end{array}
\end{align}
Board $A$ is a Latin square of $\{6,7,9\}$, and board $B$ uses only $\{6,7\}$. 
In expectation, board $A$ requires nearly $1.4$ additional rolls (in expectation) to obtain a bingo in solo play,  yet in a head-to-head, board $A$ wins more than 53\% of the time,
\begin{align}
P(T_A < T_B) \approx 0.5315, \qquad P(T_B < T_A) \approx 0.4685,
\qquad
P(T_A = T_B) = 0.
\end{align}

In [ ]:
A = [
    9, 6, 7,
    7, 9, 6,
    6, 7, 9,
]

B = [
    6, 7, 6,
    7, 7, 7,
    6, 6, 6,
]

VA = expected_winning_time(A)
VB = expected_winning_time(B)

print("Expected winning times, individual Markhov Chain")
print("\tV_A(0) =", float(VA))
print("\tV_B(0) =", float(VB))

PA, PB, PT = head_to_head_probabilities(A, B)

print("\nHead-to-head probabilities, joint Markhov Chain")
print("\tP(A wins) =", float(PA))
print("\tP(B wins) =", float(PB))
print("\tP(tie)    =", float(PT))

To generate figure 5(c),

In [ ]:
P_A, P_B, P_tie = head_to_head_distribution(A, B, kmax=50, tie_break="smallest")

# Convert fractions to floats for plotting
x = sorted(P_A.keys())
y_A = [float(P_A[k]) for k in x]
y_B = [float(P_B[k]) for k in x]
y_T = [float(P_tie[k]) for k in x]

plt.figure(figsize=(10, 5))

# Plot each distribution with markers
plt.plot(x, y_A, marker='o', linestyle='-', label='P(A wins)')
plt.plot(x, y_B, marker='s', linestyle='-', label='P(B wins)')
plt.plot(x, y_T, marker='^', linestyle='-', label='P(Tie)')

plt.xlabel("Roll number k")
plt.ylabel("Probability")
plt.title("Dice Bingo: Per-Roll Probabilities")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Nontransitive triples

We call three boards $C$, $D$, $E$ a **non-transitive triple** if in head-to-head competition, board $C$ beats board $D$, board $D$ beats board $E$, and board $E$ beats board $D$. We conducted a computational search over several hundred candidate boards and found many such triples.  Here is one example from Section 5 in the manuscript.

\begin{align}
\text{Board }C:
\begin{array}{|c|c|c|}
\hline
7 & 7 & 7 \\ \hline
6 & 6 & 6 \\ \hline
6 & 7 & 6 \\ \hline
\end{array}
\qquad
\text{Board }D: 
\begin{array}{|c|c|c|}
\hline
7 & 5 & 9 \\ \hline
9 & 7 & 4 \\ \hline
5 & 9 & 7 \\ \hline
\end{array}
\qquad
\text{Board }E: 
\begin{array}{|c|c|c|}
\hline
9 & 7 & 9 \\ \hline
9 & 9 & 9 \\ \hline
9 & 6 & 7 \\ \hline
\end{array}
\end{align}

In [ ]:
C = [
    7, 7, 7,
    6, 6, 6,
    6, 7, 6,
]

D = [
    7, 5, 9,
    9, 7, 5,
    5, 9, 7,
]

E = [
    9, 7, 9,
    9, 9, 9,
    9, 6, 7,
]

print("Solo expected times:")
print("\tV_C =", float(expected_winning_time(C)))
print("\tV_D =", float(expected_winning_time(D)))
print("\tV_E =", float(expected_winning_time(E)))

PCD_C, PCD_D, PCD_T = head_to_head_probabilities(C, D)
print("\nC vs D:")
print("\tP(C wins) =", float(PCD_C))
print("\tP(D wins) =", float(PCD_D))
print("\tP(tie)    =", float(PCD_T))

PDE_D, PDE_E, PDE_T = head_to_head_probabilities(D, E)
print("\nD vs E:")
print("\tP(D wins) =", float(PDE_D))
print("\tP(E wins) =", float(PDE_E))
print("\tP(tie)    =", float(PDE_T))

PEC_E, PEC_C, PEC_T = head_to_head_probabilities(E, C)
print("\nE vs C:")
print("\tP(E wins) =", float(PEC_E))
print("\tP(C wins) =", float(PEC_C))
print("\tP(tie)    =", float(PEC_T))
